## Block Interaction analysis

The focus of this notebook is to analyze multi-block operator replacement behaviour, in order to determine propagation error.

The following experiments work with assesment of block replacement interaction in the setting of multiple replaced blocks. The pipeline works with exclusion of first and last blocks due to strong error propagation.

In [ ]:
from dataclasses import asdict
from datetime import datetime, timezone
import json
import math
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F

In [ ]:
def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mlp_replacement.capture import collect_modules_io
from mlp_replacement.config import (
    DataConfig,
    ModelConfig,
    OperatorConfig,
    RecoveryConfig,
)
from mlp_replacement.data import build_data_loaders
from mlp_replacement.evaluation.language_model import (
    LanguageModelMetrics,
    evaluate_language_model,
)
from mlp_replacement.evaluation.operator import evaluate_operator
from mlp_replacement.model import get_mlp_block, load_model_and_tokenizer
from mlp_replacement.operators import (
    GatedMLPReplacement,
    fit_operator,
    fit_ridge_linear,
)
from mlp_replacement.compression.recovery import cache_teacher_logits, mean_cache_loss
from mlp_replacement.compression.surgery import (
    count_parameters,
    temporary_replacement,
    temporary_replacements,
)

In [ ]:
EXECUTION_MODE = 'run'
ARTIFACT_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'results'
    / 'notebook-model-study'
    / 'block-interaction.json'
)
RUN_FROM_SCRATCH = EXECUTION_MODE == 'run'
loaded_artifact = None

if not RUN_FROM_SCRATCH:
    loaded_artifact = json.loads(
        ARTIFACT_PATH.read_text(encoding='utf-8')
    )

In [ ]:
SEED = 21
torch.manual_seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
KL_RECOVERY_UPDATE_BUDGET = 64

In [ ]:
model_config = ModelConfig(
    model_id='HuggingFaceTB/SmolLM2-1.7B',
    device='auto',
    dtype='auto',
)

data_config = DataConfig(
    sequence_length=128,
    batch_size=2,
    num_calibration_batches=48,
    num_operator_validation_batches=24,
    num_recovery_batches=KL_RECOVERY_UPDATE_BUDGET,
    num_recovery_validation_batches=0,
    num_model_validation_batches=24,
    num_test_batches=0,
    seed=SEED,
)

training_config = OperatorConfig(
    epochs=64,
    learning_rate=1e-3,
    batch_size=2048,
    weight_decay=0.0,
    scheduler='constant',
    early_stopping_patience=3,
    seed=SEED,
)

recovery_config = RecoveryConfig(
    enabled=True,
    epochs=1,
    learning_rate=1e-5,
    weight_decay=0.0,
    temperature=1.0,
    cache_dtype='float16',
    early_stopping_patience=None,
)

first and last layers excluded

In [ ]:
if RUN_FROM_SCRATCH:
    model, tokenizer = load_model_and_tokenizer(model_config)
    device = next(model.parameters()).device
    ELIGIBLE_LAYERS = list(range(1, model.config.num_hidden_layers - 1))
else:
    ELIGIBLE_LAYERS = loaded_artifact['configuration']['eligible_layers']

In [ ]:
if RUN_FROM_SCRATCH:
    loaders = build_data_loaders(
        tokenizer,
        data_config,
        include_recovery=True,
    )

## Experiments

### Sliding Window Replacement analysis

- tests model degradation with rolling window replacement (e.g. 3 consecutive replacements)
- offset parameter: controls replacement offset (e.g. offset = 1 -> replacement - original - replacement)
- tests:
    - **sequential behaviour of block error propagation**
    - cummulative error across the entire model with multi-block replacement
    - block depth impact (do earlier/latter layers propagate error more?)
    - block offset / replacemnt chaining impact (does spacing between replacement helps to reduce propagated error?)
    - operator architecture placement (where to place linear layers/non-linear layers) based on depth

- ideas:
    - test linear vs non-linear operators

$S(s, k, g) = \{s, s + (g +1), s + 2(g + 1), ...\}$

where:

- $s$ start block index
- $k$ number of blocks to replace
- $g$ offset (skipping between blocks)

example:
- $s = 2, k=3, g=1 \to S = [2, 4, 6]$
- $s = 2, k=3, g=0 \to S = [2, 3, 4]$

- plots:
    - time-series lineplot: x=index, y=value
        - hue/color: linear(orange), swiglu_50(blue)
        - multiplot: first plot k=2, second plot k=3, ... stacked under each other

implementation

In [ ]:
if RUN_FROM_SCRATCH:
    WINDOW_WIDTHS = [1, 2, 3, 4]
    OPERATOR_NAMES = ['linear', 'swiglu_050']
    LINEAR_RIDGE = 1e-4
    SWIGLU_WIDTH_RATIO = 0.50
else:
    artifact_config = loaded_artifact['configuration']
    WINDOW_WIDTHS = artifact_config['window_widths']
    OPERATOR_NAMES = artifact_config['operator_names']
    LINEAR_RIDGE = artifact_config['linear_ridge']
    SWIGLU_WIDTH_RATIO = artifact_config['swiglu_width_ratio']

OPERATOR_COLORS = {
    'linear': 'tab:orange',
    'swiglu_050': 'tab:blue',
}

In [ ]:
def evaluate_cached_model(student, teacher_cache):
    was_training = student.training
    student.eval()
    kl_total = 0.0
    nll_total = 0.0
    predicted_tokens = 0

    try:
        with torch.no_grad():
            for batch in teacher_cache.batches:
                input_ids = batch.input_ids.to(device)
                attention_mask = batch.attention_mask.to(device)
                teacher_logits = batch.logits.to(device=device, dtype=torch.float32)
                student_logits = student(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                ).logits.float()

                mask = attention_mask.bool()
                temperature = recovery_config.temperature
                teacher_probabilities = torch.softmax(
                    teacher_logits[mask] / temperature,
                    dim=-1,
                )
                student_log_probabilities = torch.log_softmax(
                    student_logits[mask] / temperature,
                    dim=-1,
                )
                kl_total += float(
                    F.kl_div(
                        student_log_probabilities,
                        teacher_probabilities,
                        reduction='batchmean',
                    ).item()
                    * temperature**2
                )

                shifted_logits = student_logits[:, :-1, :].contiguous()
                shifted_labels = input_ids[:, 1:].contiguous()
                valid_mask = attention_mask[:, 1:].bool()
                shifted_labels = shifted_labels.masked_fill(~valid_mask, -100)
                nll_total += float(
                    F.cross_entropy(
                        shifted_logits.view(-1, shifted_logits.shape[-1]),
                        shifted_labels.view(-1),
                        ignore_index=-100,
                        reduction='sum',
                    ).item()
                )
                predicted_tokens += int(valid_mask.sum().item())
    finally:
        student.train(was_training)

    loss = nll_total / predicted_tokens
    return {
        'teacher_kl': kl_total / len(teacher_cache),
        'model_loss': loss,
        'perplexity': math.exp(loss) if loss < 709 else float('inf'),
    }

In [ ]:
if RUN_FROM_SCRATCH:
    blocks = {layer: get_mlp_block(model, layer) for layer in ELIGIBLE_LAYERS}
    target_paths = [blocks[layer].path for layer in ELIGIBLE_LAYERS]
    training_pairs = collect_modules_io(
        model,
        target_paths,
        loaders.calibration,
        data_config.num_calibration_batches,
        device,
    )
    validation_pairs = collect_modules_io(
        model,
        target_paths,
        loaders.operator_validation,
        data_config.num_operator_validation_batches,
        device,
    )
    dense_metrics = evaluate_language_model(
        model,
        loaders.model_validation,
        device,
        data_config.num_model_validation_batches,
    )
    teacher_cache = cache_teacher_logits(
        model,
        loaders.model_validation,
        data_config.num_model_validation_batches,
        device,
        recovery_config.cache_dtype,
    )

    fitted_operators = {name: {} for name in OPERATOR_NAMES}
    operator_histories = {}
    operator_fit_rows = []

    for layer in ELIGIBLE_LAYERS:
        block = blocks[layer]
        train_pairs = training_pairs[block.path]
        val_pairs = validation_pairs[block.path]

        linear = fit_ridge_linear(
            train_pairs,
            ridge=LINEAR_RIDGE,
            bias=False,
            device=device,
        ).to(device)
        linear_metrics = evaluate_operator(
            linear,
            val_pairs,
            device,
            training_config.batch_size,
        )
        operator_fit_rows.append({
            'layer': layer,
            'operator': 'linear',
            'parameters': count_parameters(linear),
            'validation_mse': linear_metrics.mse,
            'validation_relative_mse': linear_metrics.relative_mse,
            'best_epoch': 0,
        })
        fitted_operators['linear'][layer] = linear.to('cpu')

        replacement_width = max(
            1,
            round(block.module.up_proj.out_features * SWIGLU_WIDTH_RATIO),
        )
        torch.manual_seed(training_config.seed)
        swiglu_fit = fit_operator(
            GatedMLPReplacement(
                hidden_size=train_pairs.hidden_size,
                bottleneck_size=replacement_width,
                bias=False,
            ),
            train_pairs,
            val_pairs,
            training_config,
            device,
        )
        swiglu_metrics = evaluate_operator(
            swiglu_fit.module,
            val_pairs,
            device,
            training_config.batch_size,
        )
        operator_fit_rows.append({
            'layer': layer,
            'operator': 'swiglu_050',
            'parameters': count_parameters(swiglu_fit.module),
            'validation_mse': swiglu_metrics.mse,
            'validation_relative_mse': swiglu_metrics.relative_mse,
            'best_epoch': swiglu_fit.best_epoch,
        })
        operator_histories[(layer, 'swiglu_050')] = [
            asdict(epoch) for epoch in swiglu_fit.history
        ]
        fitted_operators['swiglu_050'][layer] = swiglu_fit.module.to('cpu')

    operator_fit_df = pd.DataFrame(operator_fit_rows)

    window_rows = []
    for operator_name in OPERATOR_NAMES:
        for window_width in WINDOW_WIDTHS:
            for start_position in range(len(ELIGIBLE_LAYERS) - window_width + 1):
                window_layers = tuple(
                    ELIGIBLE_LAYERS[start_position:start_position + window_width]
                )
                replacements = {
                    layer: fitted_operators[operator_name][layer]
                    for layer in window_layers
                }
                with temporary_replacements(model, replacements):
                    metrics = evaluate_cached_model(model, teacher_cache)
                for replacement in replacements.values():
                    replacement.to('cpu')

                window_rows.append({
                    'operator': operator_name,
                    'window_width': window_width,
                    'window_start': window_layers[0],
                    'window_end': window_layers[-1],
                    'layers': window_layers,
                    **metrics,
                })
            print(f'{operator_name}, w={window_width}: complete')

    window_df = pd.DataFrame(window_rows)
    singleton_kl = {
        (row.operator, row.window_start): row.teacher_kl
        for row in window_df.itertuples()
        if row.window_width == 1
    }
    window_df['additive_singleton_kl'] = window_df.apply(
        lambda row: sum(
            singleton_kl[(row['operator'], layer)]
            for layer in row['layers']
        ),
        axis=1,
    )
    window_df['excess_kl'] = (
        window_df['teacher_kl'] - window_df['additive_singleton_kl']
    )
else:
    dense_metrics = LanguageModelMetrics(
        **loaded_artifact['results']['dense_language_model']
    )
    operator_fit_df = pd.DataFrame(
        loaded_artifact['results']['operator_fitting']
    )
    operator_histories = {
        (record['layer'], record['operator']): record['history']
        for record in loaded_artifact['results']['operator_training_history']
    }
    window_df = pd.DataFrame(
        loaded_artifact['results']['sliding_windows']
    )

In [ ]:
figure, axes = plt.subplots(
    len(WINDOW_WIDTHS),
    2,
    figsize=(15, 11),
    sharex='col',
)

for row_index, window_width in enumerate(WINDOW_WIDTHS):
    subset = window_df[window_df['window_width'] == window_width]
    kl_axis, ppl_axis = axes[row_index]

    sns.lineplot(
        data=subset,
        x='window_start',
        y='teacher_kl',
        hue='operator',
        palette=OPERATOR_COLORS,
        marker='o',
        legend=row_index == 0,
        ax=kl_axis,
    )
    sns.lineplot(
        data=subset,
        x='window_start',
        y='perplexity',
        hue='operator',
        palette=OPERATOR_COLORS,
        marker='o',
        legend=False,
        ax=ppl_axis,
    )
    ppl_axis.axhline(
        dense_metrics.perplexity,
        color='black',
        linestyle='--',
        linewidth=1,
    )
    ppl_axis.set_yscale('log')
    kl_axis.set_ylabel(f'w={window_width}\nKL')
    ppl_axis.set_ylabel(f'w={window_width}\nPPL')
    if row_index < len(WINDOW_WIDTHS) - 1:
        kl_axis.set_xlabel('')
        ppl_axis.set_xlabel('')

axes[0, 0].set_title('Teacher-to-student KL divergence')
axes[0, 1].set_title('Perplexity (log scale)')
axes[-1, 0].set_xlabel('Window start layer')
axes[-1, 1].set_xlabel('Window start layer')
figure.suptitle('Consecutive replacement windows')
figure.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

interaction_figure, interaction_axes = plt.subplots(
    len(WINDOW_WIDTHS),
    1,
    figsize=(15, 8),
    sharex=True,
)

for row_index, window_width in enumerate(WINDOW_WIDTHS):
    axis = interaction_axes[row_index]
    subset = window_df[window_df['window_width'] == window_width]
    sns.lineplot(
        data=subset,
        x='window_start',
        y='excess_kl',
        hue='operator',
        palette=OPERATOR_COLORS,
        marker='o',
        legend=row_index == 0,
        ax=axis,
    )
    lower, upper = axis.get_ylim()
    if upper > 0:
        axis.axhspan(0, upper, color='red', alpha=0.06, zorder=0)
    if lower < 0:
        axis.axhspan(lower, 0, color='tab:blue', alpha=0.06, zorder=0)
    axis.axhline(0, color='black', linestyle='--', linewidth=1)
    axis.set_ylabel(f'w={window_width}\n$E_w(s)$')
    if row_index < len(WINDOW_WIDTHS) - 1:
        axis.set_xlabel('')

interaction_axes[-1].set_xlabel('Window start layer')
interaction_figure.suptitle(
    'Excess KL beyond singleton additivity ' 
    '(red: amplification, blue: attenuation)'
)
interaction_figure.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

### Operator Block Interaction

#### Pairwise interaction analysis
- using KL divergence tests:
    - block $i$ using $KL_i$
    - block $j$ using $KL_j$
    - replaced blocks using $K_{ij}$

- tests:
    - $I_{ij} = K_{ij} - (K_i + K_j)$
    - tests error cummulation linearity
    - $d$ distance between blocks $d = |i-j|$
    - tests proximity interaction
    - cross-model compatibility version $d_{norm} = \frac{|i-j|}{L-1}$

- metrics:
    - KL div
    - PPL

- evaluation:
    - if $I_{ij} \approx 0$ -> replacements propagate error additively (idependent?)
    - if $I_{ij} > 0$ -> replacements amplify error propagation
    - if $I_{ij} < 0 $ -> replacement damage is less than expected

- plots:
    - relational matrix heatmap where val=I_ij (full matrix to see ij vs ji)

- problems:
    - problem: $I_{ij}$ represents replacements of both i,j with indentical operators, different operators would be $I_{ij}^{a,b}$ where a,b are replacement operators of i,j
    - solution: I guess we can report this just with a simple table or map of matrices (ordered small matrices in grid, one matrix one operator combination)
    

#### higher order interaction

Use inclusion/exclusion principle for scaling.



bonus ideas:
- Zeby sa tu dal mozno vyuzit cross quantilogram (CQ) pre testovanie toho impactu medzi blokovo? (to by bolo dost zaujimave takato bakalarka aplikacia). Lag operator by umoznoval sledovat propagaciu erroru maybe, a vizualizovat to cez network graf kruhovy

tests:
- $S$ = index set (e.g. i, j, k)
- if $K_S$ satisfies $K_S = \sum{K_i}$ (simple sum)
- if $K_S$ follows Inclusion-exclusion sum

##### pairwise

homogenous

In [ ]:
if RUN_FROM_SCRATCH:
    PAIRWISE_ELIGIBLE_LAYERS = list(ELIGIBLE_LAYERS)
    PAIRWISE_OPERATOR_NAMES = ['linear', 'swiglu_050']
    PAIRWISE_HETEROGENEOUS_OPERATOR_PAIRS = [
        ('linear', 'swiglu_050')
    ]
    PAIRWISE_LINEAR_RIDGE = 1e-4
    PAIRWISE_SWIGLU_WIDTH_RATIO = 0.50
    PAIRWISE_DEPTH_DENOMINATOR = model.config.num_hidden_layers - 1
else:
    pairwise_config = loaded_artifact['configuration']['pairwise_interaction']
    PAIRWISE_ELIGIBLE_LAYERS = pairwise_config['eligible_layers']
    PAIRWISE_OPERATOR_NAMES = pairwise_config['operator_names']
    PAIRWISE_HETEROGENEOUS_OPERATOR_PAIRS = [
        tuple(pair) for pair in pairwise_config['heterogeneous_operator_pairs']
    ]
    PAIRWISE_LINEAR_RIDGE = pairwise_config['linear_ridge']
    PAIRWISE_SWIGLU_WIDTH_RATIO = pairwise_config['swiglu_width_ratio']
    PAIRWISE_DEPTH_DENOMINATOR = pairwise_config['depth_denominator']

PAIRWISE_OPERATOR_COLORS = {
    'linear': 'tab:orange',
    'swiglu_050': 'tab:blue',
}

In [ ]:
def evaluate_pairwise_cached_model(student, teacher_cache):
    was_training = student.training
    student.eval()
    kl_total = 0.0
    nll_total = 0.0
    predicted_tokens = 0

    try:
        with torch.no_grad():
            for batch in teacher_cache.batches:
                input_ids = batch.input_ids.to(device)
                attention_mask = batch.attention_mask.to(device)
                teacher_logits = batch.logits.to(
                    device=device, dtype=torch.float32
                )
                student_logits = student(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                ).logits.float()

                mask = attention_mask.bool()
                temperature = recovery_config.temperature
                teacher_probabilities = torch.softmax(
                    teacher_logits[mask] / temperature,
                    dim=-1,
                )
                student_log_probabilities = torch.log_softmax(
                    student_logits[mask] / temperature,
                    dim=-1,
                )
                kl_total += float(
                    F.kl_div(
                        student_log_probabilities,
                        teacher_probabilities,
                        reduction='batchmean',
                    ).item()
                    * temperature**2
                )

                shifted_logits = student_logits[:, :-1, :].contiguous()
                shifted_labels = input_ids[:, 1:].contiguous()
                valid_mask = attention_mask[:, 1:].bool()
                shifted_labels = shifted_labels.masked_fill(~valid_mask, -100)
                nll_total += float(
                    F.cross_entropy(
                        shifted_logits.view(-1, shifted_logits.shape[-1]),
                        shifted_labels.view(-1),
                        ignore_index=-100,
                        reduction='sum',
                    ).item()
                )
                predicted_tokens += int(valid_mask.sum().item())
    finally:
        student.train(was_training)

    loss = nll_total / predicted_tokens
    return {
        'teacher_kl': kl_total / len(teacher_cache),
        'model_loss': loss,
        'perplexity': math.exp(loss) if loss < 709 else float('inf'),
    }

In [ ]:
if RUN_FROM_SCRATCH:
    pairwise_blocks = {
        layer: get_mlp_block(model, layer)
        for layer in PAIRWISE_ELIGIBLE_LAYERS
    }
    pairwise_target_paths = [
        pairwise_blocks[layer].path
        for layer in PAIRWISE_ELIGIBLE_LAYERS
    ]
    pairwise_training_pairs = collect_modules_io(
        model,
        pairwise_target_paths,
        loaders.calibration,
        data_config.num_calibration_batches,
        device,
    )
    pairwise_validation_pairs = collect_modules_io(
        model,
        pairwise_target_paths,
        loaders.operator_validation,
        data_config.num_operator_validation_batches,
        device,
    )
    pairwise_dense_metrics = evaluate_language_model(
        model,
        loaders.model_validation,
        device,
        data_config.num_model_validation_batches,
    )
    pairwise_teacher_cache = cache_teacher_logits(
        model,
        loaders.model_validation,
        data_config.num_model_validation_batches,
        device,
        recovery_config.cache_dtype,
    )

    pairwise_fitted_operators = {
        name: {} for name in PAIRWISE_OPERATOR_NAMES
    }
    pairwise_operator_histories = {}
    pairwise_operator_fit_rows = []

    for layer in PAIRWISE_ELIGIBLE_LAYERS:
        block = pairwise_blocks[layer]
        train_pairs = pairwise_training_pairs[block.path]
        val_pairs = pairwise_validation_pairs[block.path]

        linear = fit_ridge_linear(
            train_pairs,
            ridge=PAIRWISE_LINEAR_RIDGE,
            bias=False,
            device=device,
        ).to(device)
        linear_metrics = evaluate_operator(
            linear,
            val_pairs,
            device,
            training_config.batch_size,
        )
        pairwise_operator_fit_rows.append({
            'layer': layer,
            'operator': 'linear',
            'parameters': count_parameters(linear),
            'validation_mse': linear_metrics.mse,
            'validation_relative_mse': linear_metrics.relative_mse,
            'best_epoch': 0,
        })
        pairwise_fitted_operators['linear'][layer] = linear.to('cpu')

        replacement_width = max(
            1,
            round(
                block.module.up_proj.out_features
                * PAIRWISE_SWIGLU_WIDTH_RATIO
            ),
        )
        torch.manual_seed(training_config.seed)
        swiglu_fit = fit_operator(
            GatedMLPReplacement(
                hidden_size=train_pairs.hidden_size,
                bottleneck_size=replacement_width,
                bias=False,
            ),
            train_pairs,
            val_pairs,
            training_config,
            device,
        )
        swiglu_metrics = evaluate_operator(
            swiglu_fit.module,
            val_pairs,
            device,
            training_config.batch_size,
        )
        pairwise_operator_fit_rows.append({
            'layer': layer,
            'operator': 'swiglu_050',
            'parameters': count_parameters(swiglu_fit.module),
            'validation_mse': swiglu_metrics.mse,
            'validation_relative_mse': swiglu_metrics.relative_mse,
            'best_epoch': swiglu_fit.best_epoch,
        })
        pairwise_operator_histories[(layer, 'swiglu_050')] = [
            asdict(epoch) for epoch in swiglu_fit.history
        ]
        pairwise_fitted_operators['swiglu_050'][layer] = (
            swiglu_fit.module.to('cpu')
        )

    pairwise_operator_fit_df = pd.DataFrame(
        pairwise_operator_fit_rows
    )
    del pairwise_training_pairs, pairwise_validation_pairs

    pairwise_parameter_lookup = {
        (row.operator, row.layer): row.parameters
        for row in pairwise_operator_fit_df.itertuples()
    }
    pairwise_singleton_rows = []
    for operator_name in PAIRWISE_OPERATOR_NAMES:
        for layer in PAIRWISE_ELIGIBLE_LAYERS:
            replacement = pairwise_fitted_operators[operator_name][layer]
            with temporary_replacement(model, layer, replacement):
                metrics = evaluate_pairwise_cached_model(
                    model, pairwise_teacher_cache
                )
            replacement.to('cpu')
            pairwise_singleton_rows.append({
                'operator': operator_name,
                'layer': layer,
                'parameters': pairwise_parameter_lookup[(operator_name, layer)],
                'teacher_kl': metrics['teacher_kl'],
                'model_loss': metrics['model_loss'],
                'perplexity': metrics['perplexity'],
                'perplexity_delta': (
                    metrics['perplexity']
                    - pairwise_dense_metrics.perplexity
                ),
            })
        print(f'{operator_name}: singleton profiling complete')

    pairwise_singleton_df = pd.DataFrame(pairwise_singleton_rows)
else:
    pairwise_results = loaded_artifact['results']['pairwise_interaction']
    pairwise_dense_metrics = LanguageModelMetrics(
        **pairwise_results['dense_language_model']
    )
    pairwise_operator_fit_df = pd.DataFrame(
        pairwise_results['operator_fitting']
    )
    pairwise_operator_histories = {
        (record['layer'], record['operator']): record['history']
        for record in pairwise_results['operator_training_history']
    }
    pairwise_singleton_df = pd.DataFrame(
        pairwise_results['singletons']
    )

pairwise_singleton_kl = {
    (row.operator, row.layer): row.teacher_kl
    for row in pairwise_singleton_df.itertuples()
}
pairwise_singleton_parameters = {
    (row.operator, row.layer): row.parameters
    for row in pairwise_singleton_df.itertuples()
}

In [ ]:
if RUN_FROM_SCRATCH:
    homogeneous_pair_rows = []

    for operator_name in PAIRWISE_OPERATOR_NAMES:
        for left_position, layer_i in enumerate(PAIRWISE_ELIGIBLE_LAYERS):
            for layer_j in PAIRWISE_ELIGIBLE_LAYERS[left_position + 1:]:
                replacements = {
                    layer_i: pairwise_fitted_operators[operator_name][layer_i],
                    layer_j: pairwise_fitted_operators[operator_name][layer_j],
                }
                with temporary_replacements(model, replacements):
                    metrics = evaluate_pairwise_cached_model(
                        model, pairwise_teacher_cache
                    )
                for replacement in replacements.values():
                    replacement.to('cpu')

                singleton_kl_i = pairwise_singleton_kl[(operator_name, layer_i)]
                singleton_kl_j = pairwise_singleton_kl[(operator_name, layer_j)]
                additive_kl = singleton_kl_i + singleton_kl_j
                interaction_kl = metrics['teacher_kl'] - additive_kl
                parameters_i = pairwise_singleton_parameters[
                    (operator_name, layer_i)
                ]
                parameters_j = pairwise_singleton_parameters[
                    (operator_name, layer_j)
                ]
                distance = abs(layer_i - layer_j)

                homogeneous_pair_rows.append({
                    'interaction_type': 'homogeneous',
                    'operator_condition': (
                        f'{operator_name}__{operator_name}'
                    ),
                    'operator_i': operator_name,
                    'operator_j': operator_name,
                    'layer_i': layer_i,
                    'layer_j': layer_j,
                    'distance': distance,
                    'normalized_distance': (
                        distance / PAIRWISE_DEPTH_DENOMINATOR
                    ),
                    'parameters_i': parameters_i,
                    'parameters_j': parameters_j,
                    'pair_parameters': parameters_i + parameters_j,
                    'singleton_kl_i': singleton_kl_i,
                    'singleton_kl_j': singleton_kl_j,
                    'additive_singleton_kl': additive_kl,
                    'pair_kl': metrics['teacher_kl'],
                    'interaction_kl': interaction_kl,
                    'absolute_interaction_kl': abs(interaction_kl),
                    'model_loss': metrics['model_loss'],
                    'perplexity': metrics['perplexity'],
                    'perplexity_delta': (
                        metrics['perplexity']
                        - pairwise_dense_metrics.perplexity
                    ),
                })
        print(f'{operator_name}: homogeneous pairs complete')

    homogeneous_pair_df = pd.DataFrame(homogeneous_pair_rows)
else:
    homogeneous_pair_df = pd.DataFrame(
        pairwise_results['homogeneous_pairs']
    )

In [ ]:
homogeneous_reporting_df = homogeneous_pair_df.assign(
    amplifying=homogeneous_pair_df['interaction_kl'] > 0
)
homogeneous_summary_df = (
    homogeneous_reporting_df
    .groupby('operator_i', as_index=False)
    .agg(
        pair_count=('interaction_kl', 'size'),
        mean_pair_kl=('pair_kl', 'mean'),
        median_interaction_kl=('interaction_kl', 'median'),
        mean_absolute_interaction_kl=(
            'absolute_interaction_kl', 'mean'
        ),
        maximum_amplification_kl=('interaction_kl', 'max'),
        maximum_attenuation_kl=('interaction_kl', 'min'),
        amplification_fraction=('amplifying', 'mean'),
        mean_perplexity=('perplexity', 'mean'),
        maximum_perplexity=('perplexity', 'max'),
    )
    .rename(columns={'operator_i': 'operator'})
)

homogeneous_amplification_df = (
    homogeneous_pair_df
    .sort_values(
        ['operator_i', 'interaction_kl'],
        ascending=[True, False],
    )
    .groupby('operator_i', sort=False)
    .head(5)
    .assign(extreme='amplification')
)
homogeneous_attenuation_df = (
    homogeneous_pair_df
    .sort_values(
        ['operator_i', 'interaction_kl'],
        ascending=[True, True],
    )
    .groupby('operator_i', sort=False)
    .head(5)
    .assign(extreme='attenuation')
)
homogeneous_extremes_df = pd.concat(
    [homogeneous_amplification_df, homogeneous_attenuation_df],
    ignore_index=True,
)[
    [
        'extreme',
        'operator_i',
        'layer_i',
        'layer_j',
        'distance',
        'pair_kl',
        'interaction_kl',
        'perplexity',
        'perplexity_delta',
    ]
]

display(homogeneous_summary_df)
display(homogeneous_extremes_df)

homogeneous_interaction_matrices = {}
for operator_name in PAIRWISE_OPERATOR_NAMES:
    matrix = pd.DataFrame(
        index=PAIRWISE_ELIGIBLE_LAYERS,
        columns=PAIRWISE_ELIGIBLE_LAYERS,
        dtype=float,
    )
    subset = homogeneous_pair_df[
        homogeneous_pair_df['operator_i'] == operator_name
    ]
    for row in subset.itertuples():
        matrix.loc[row.layer_i, row.layer_j] = row.interaction_kl
        matrix.loc[row.layer_j, row.layer_i] = row.interaction_kl
    homogeneous_interaction_matrices[operator_name] = matrix

homogeneous_color_limit = max(
    float(homogeneous_pair_df['absolute_interaction_kl'].max()),
    1e-12,
)
figure, axes = plt.subplots(
    1,
    len(PAIRWISE_OPERATOR_NAMES),
    figsize=(14, 6),
)
if len(PAIRWISE_OPERATOR_NAMES) == 1:
    axes = [axes]

for axis, operator_name in zip(
    axes, PAIRWISE_OPERATOR_NAMES, strict=True
):
    sns.heatmap(
        homogeneous_interaction_matrices[operator_name],
        cmap='coolwarm',
        center=0,
        vmin=-homogeneous_color_limit,
        vmax=homogeneous_color_limit,
        square=True,
        cbar_kws={'label': r'$I_{ij}$'},
        ax=axis,
    )
    axis.set_title(f'{operator_name} x {operator_name}')
    axis.set_xlabel('Layer j')
    axis.set_ylabel('Layer i')

figure.suptitle('Homogeneous pairwise KL interaction')
figure.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

heterogenous

In [ ]:
if RUN_FROM_SCRATCH:
    heterogeneous_pair_rows = []

    for operator_i, operator_j in PAIRWISE_HETEROGENEOUS_OPERATOR_PAIRS:
        for layer_i in PAIRWISE_ELIGIBLE_LAYERS:
            for layer_j in PAIRWISE_ELIGIBLE_LAYERS:
                if layer_i == layer_j:
                    continue

                replacements = {
                    layer_i: pairwise_fitted_operators[operator_i][layer_i],
                    layer_j: pairwise_fitted_operators[operator_j][layer_j],
                }
                with temporary_replacements(model, replacements):
                    metrics = evaluate_pairwise_cached_model(
                        model, pairwise_teacher_cache
                    )
                for replacement in replacements.values():
                    replacement.to('cpu')

                singleton_kl_i = pairwise_singleton_kl[(operator_i, layer_i)]
                singleton_kl_j = pairwise_singleton_kl[(operator_j, layer_j)]
                additive_kl = singleton_kl_i + singleton_kl_j
                interaction_kl = metrics['teacher_kl'] - additive_kl
                parameters_i = pairwise_singleton_parameters[
                    (operator_i, layer_i)
                ]
                parameters_j = pairwise_singleton_parameters[
                    (operator_j, layer_j)
                ]
                distance = abs(layer_i - layer_j)

                heterogeneous_pair_rows.append({
                    'interaction_type': 'heterogeneous',
                    'operator_condition': f'{operator_i}__{operator_j}',
                    'operator_i': operator_i,
                    'operator_j': operator_j,
                    'layer_i': layer_i,
                    'layer_j': layer_j,
                    'distance': distance,
                    'normalized_distance': (
                        distance / PAIRWISE_DEPTH_DENOMINATOR
                    ),
                    'parameters_i': parameters_i,
                    'parameters_j': parameters_j,
                    'pair_parameters': parameters_i + parameters_j,
                    'singleton_kl_i': singleton_kl_i,
                    'singleton_kl_j': singleton_kl_j,
                    'additive_singleton_kl': additive_kl,
                    'pair_kl': metrics['teacher_kl'],
                    'interaction_kl': interaction_kl,
                    'absolute_interaction_kl': abs(interaction_kl),
                    'model_loss': metrics['model_loss'],
                    'perplexity': metrics['perplexity'],
                    'perplexity_delta': (
                        metrics['perplexity']
                        - pairwise_dense_metrics.perplexity
                    ),
                })
        print(f'{operator_i} x {operator_j}: heterogeneous pairs complete')

    heterogeneous_pair_df = pd.DataFrame(heterogeneous_pair_rows)
else:
    heterogeneous_pair_df = pd.DataFrame(
        pairwise_results['heterogeneous_pairs']
    )

In [ ]:
heterogeneous_reporting_df = heterogeneous_pair_df.assign(
    amplifying=heterogeneous_pair_df['interaction_kl'] > 0
)
heterogeneous_summary_df = (
    heterogeneous_reporting_df
    .groupby('operator_condition', as_index=False)
    .agg(
        pair_count=('interaction_kl', 'size'),
        mean_pair_kl=('pair_kl', 'mean'),
        median_interaction_kl=('interaction_kl', 'median'),
        mean_absolute_interaction_kl=(
            'absolute_interaction_kl', 'mean'
        ),
        maximum_amplification_kl=('interaction_kl', 'max'),
        maximum_attenuation_kl=('interaction_kl', 'min'),
        amplification_fraction=('amplifying', 'mean'),
        mean_perplexity=('perplexity', 'mean'),
        maximum_perplexity=('perplexity', 'max'),
    )
)

heterogeneous_amplification_df = (
    heterogeneous_pair_df
    .sort_values(
        ['operator_condition', 'interaction_kl'],
        ascending=[True, False],
    )
    .groupby('operator_condition', sort=False)
    .head(5)
    .assign(extreme='amplification')
)
heterogeneous_attenuation_df = (
    heterogeneous_pair_df
    .sort_values(
        ['operator_condition', 'interaction_kl'],
        ascending=[True, True],
    )
    .groupby('operator_condition', sort=False)
    .head(5)
    .assign(extreme='attenuation')
)
heterogeneous_extremes_df = pd.concat(
    [heterogeneous_amplification_df, heterogeneous_attenuation_df],
    ignore_index=True,
)[
    [
        'extreme',
        'operator_condition',
        'layer_i',
        'layer_j',
        'distance',
        'pair_kl',
        'interaction_kl',
        'perplexity',
        'perplexity_delta',
    ]
]

display(heterogeneous_summary_df)
display(heterogeneous_extremes_df)

heterogeneous_interaction_matrices = {}
for operator_i, operator_j in PAIRWISE_HETEROGENEOUS_OPERATOR_PAIRS:
    condition = f'{operator_i}__{operator_j}'
    matrix = (
        heterogeneous_pair_df[
            heterogeneous_pair_df['operator_condition'] == condition
        ]
        .pivot(
            index='layer_i',
            columns='layer_j',
            values='interaction_kl',
        )
        .reindex(
            index=PAIRWISE_ELIGIBLE_LAYERS,
            columns=PAIRWISE_ELIGIBLE_LAYERS,
        )
    )
    heterogeneous_interaction_matrices[condition] = matrix

heterogeneous_color_limit = max(
    float(heterogeneous_pair_df['absolute_interaction_kl'].max()),
    1e-12,
)
figure, axes = plt.subplots(
    1,
    len(PAIRWISE_HETEROGENEOUS_OPERATOR_PAIRS),
    figsize=(7 * len(PAIRWISE_HETEROGENEOUS_OPERATOR_PAIRS), 6),
)
if len(PAIRWISE_HETEROGENEOUS_OPERATOR_PAIRS) == 1:
    axes = [axes]

for axis, (operator_i, operator_j) in zip(
    axes,
    PAIRWISE_HETEROGENEOUS_OPERATOR_PAIRS,
    strict=True,
):
    condition = f'{operator_i}__{operator_j}'
    sns.heatmap(
        heterogeneous_interaction_matrices[condition],
        cmap='coolwarm',
        center=0,
        vmin=-heterogeneous_color_limit,
        vmax=heterogeneous_color_limit,
        square=True,
        cbar_kws={'label': r'$I_{ij}$'},
        ax=axis,
    )
    axis.set_title(f'{operator_i} at i x {operator_j} at j')
    axis.set_xlabel(f'Layer j ({operator_j})')
    axis.set_ylabel(f'Layer i ({operator_i})')

figure.suptitle('Heterogeneous pairwise KL interaction')
figure.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

pairwise_distance_df = pd.concat(
    [
        homogeneous_pair_df.assign(
            condition=(
                homogeneous_pair_df['operator_i']
                + ' x '
                + homogeneous_pair_df['operator_j']
            )
        ),
        heterogeneous_pair_df.assign(
            condition=(
                heterogeneous_pair_df['operator_i']
                + ' x '
                + heterogeneous_pair_df['operator_j']
            )
        ),
    ],
    ignore_index=True,
)
pairwise_distance_summary_df = (
    pairwise_distance_df
    .groupby(['interaction_type', 'condition', 'distance'], as_index=False)
    .agg(
        pair_count=('interaction_kl', 'size'),
        median_interaction_kl=('interaction_kl', 'median'),
        median_absolute_interaction_kl=(
            'absolute_interaction_kl', 'median'
        ),
        mean_absolute_interaction_kl=(
            'absolute_interaction_kl', 'mean'
        ),
        mean_pair_kl=('pair_kl', 'mean'),
        mean_perplexity=('perplexity', 'mean'),
    )
)

display(pairwise_distance_summary_df)

figure, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(
    data=pairwise_distance_summary_df,
    x='distance',
    y='median_interaction_kl',
    hue='condition',
    marker='o',
    ax=axes[0],
)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Signed interaction by layer distance')
axes[0].set_ylabel(r'Median $I_{ij}$')

sns.lineplot(
    data=pairwise_distance_summary_df,
    x='distance',
    y='median_absolute_interaction_kl',
    hue='condition',
    marker='o',
    ax=axes[1],
)
axes[1].set_title('Interaction magnitude by layer distance')
axes[1].set_ylabel(r'Median $|I_{ij}|$')
figure.tight_layout()
plt.show()

##### higher-order

### Results

implications:
- Jut like modeGPT we want to use the determined importance I(l) statistic to compute block sparsity allocation

In [ ]:
def json_records(frame):
    return json.loads(
        frame.to_json(orient='records', double_precision=15)
    )


if RUN_FROM_SCRATCH:
    artifact = {
        'schema_version': 2,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'configuration': {
            'model': {
                **asdict(model_config),
                'resolved_revision': getattr(
                    model.config, '_commit_hash', None
                ),
            },
            'data': asdict(data_config),
            'operator_training': asdict(training_config),
            'evaluation': {
                'temperature': recovery_config.temperature,
                'teacher_cache_dtype': recovery_config.cache_dtype,
                'recovery_performed': False,
            },
            'eligible_layers': list(ELIGIBLE_LAYERS),
            'excluded_layers': [
                0, model.config.num_hidden_layers - 1
            ],
            'window_widths': list(WINDOW_WIDTHS),
            'operator_names': list(OPERATOR_NAMES),
            'linear_ridge': LINEAR_RIDGE,
            'swiglu_width_ratio': SWIGLU_WIDTH_RATIO,
            'pairwise_interaction': {
                'eligible_layers': list(PAIRWISE_ELIGIBLE_LAYERS),
                'operator_names': list(PAIRWISE_OPERATOR_NAMES),
                'heterogeneous_operator_pairs': [
                    list(pair)
                    for pair in PAIRWISE_HETEROGENEOUS_OPERATOR_PAIRS
                ],
                'linear_ridge': PAIRWISE_LINEAR_RIDGE,
                'swiglu_width_ratio': PAIRWISE_SWIGLU_WIDTH_RATIO,
                'depth_denominator': PAIRWISE_DEPTH_DENOMINATOR,
                'interaction_metric': 'teacher_kl_additivity_residual',
                'budget_matched': False,
                'recovery_performed': False,
            },
        },
        'results': {
            'dense_language_model': asdict(dense_metrics),
            'operator_fitting': json_records(operator_fit_df),
            'operator_training_history': [
                {
                    'layer': layer,
                    'operator': operator,
                    'history': history,
                }
                for (layer, operator), history
                in operator_histories.items()
            ],
            'sliding_windows': json_records(window_df),
            'pairwise_interaction': {
                'dense_language_model': asdict(
                    pairwise_dense_metrics
                ),
                'operator_fitting': json_records(
                    pairwise_operator_fit_df
                ),
                'operator_training_history': [
                    {
                        'layer': layer,
                        'operator': operator,
                        'history': history,
                    }
                    for (layer, operator), history
                    in pairwise_operator_histories.items()
                ],
                'singletons': json_records(pairwise_singleton_df),
                'homogeneous_pairs': json_records(
                    homogeneous_pair_df
                ),
                'homogeneous_summary': json_records(
                    homogeneous_summary_df
                ),
                'homogeneous_extremes': json_records(
                    homogeneous_extremes_df
                ),
                'heterogeneous_pairs': json_records(
                    heterogeneous_pair_df
                ),
                'heterogeneous_summary': json_records(
                    heterogeneous_summary_df
                ),
                'heterogeneous_extremes': json_records(
                    heterogeneous_extremes_df
                ),
                'distance_summary': json_records(
                    pairwise_distance_summary_df
                ),
            },
        },
    }
    ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
    ARTIFACT_PATH.write_text(
        json.dumps(artifact, indent=2, allow_nan=False),
        encoding='utf-8',
    )
    print(f'Saved block-interaction artifact to {ARTIFACT_PATH}')
else:
    artifact = loaded_artifact
    print(f'Loaded block-interaction artifact from {ARTIFACT_PATH}')